# 06 — GNN Message Passing + Backprop Experiment

Petit sandbox pour tester si un entrainement GNN (backprop numerique)
ameliore un signal structurel local (similarite parent/enfant).

Ce notebook ne modifie pas les params GNN en base: tout se fait en memoire.

In [1]:
import { openVaultStore } from "../src/db/index.ts";
import { parseVault } from "../src/parser.ts";
import { DenoVaultReader } from "../src/io.ts";
import { gnnForward } from "../src/gnn/forward.ts";
import { initParams } from "../src/gnn/params.ts";
import { gnnTrainStep } from "../src/gnn/backward.ts";
import { getBlasStatus, isBlasAvailable } from "../src/gnn/blas-ffi.ts";
import type { GNNNode, GNNConfig } from "../src/gnn/types.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const db = await openVaultStore(DB_PATH);
const rows = await db.getAllNotes();
const notes = await parseVault(new DenoVaultReader(), VAULT_PATH);

const embNameSet = new Set(rows.filter((r) => r.embedding).map((r) => r.name));
const noteByName = new Map(notes.map((n) => [n.name, n]));

const undirected = new Map<string, Set<string>>();
for (const n of notes) {
  if (!embNameSet.has(n.name)) continue;
  undirected.set(n.name, undirected.get(n.name) ?? new Set());
  for (const c of n.wikilinks) {
    if (!embNameSet.has(c)) continue;
    (undirected.get(n.name)!).add(c);
    if (!undirected.has(c)) undirected.set(c, new Set());
    (undirected.get(c)!).add(n.name);
  }
}

function pickSubset(minSize = 6, maxSize = 10): string[] {
  const starts = [...undirected.entries()]
    .filter(([, nbrs]) => nbrs.size > 0)
    .sort((a, b) => b[1].size - a[1].size)
    .map(([name]) => name);

  for (const start of starts) {
    const queue = [start];
    const seen = new Set<string>();
    const comp: string[] = [];
    while (queue.length > 0) {
      const cur = queue.shift()!;
      if (seen.has(cur)) continue;
      seen.add(cur);
      comp.push(cur);
      for (const nxt of undirected.get(cur) ?? []) {
        if (!seen.has(nxt)) queue.push(nxt);
      }
    }
    if (comp.length >= minSize) return comp.slice(0, maxSize);
  }

  return starts.slice(0, Math.max(4, Math.min(maxSize, starts.length)));
}

const picked = pickSubset();
const subset = rows.filter((r) => picked.includes(r.name) && r.embedding);
if (subset.length < 4) throw new Error("No connected subset with embeddings found");

const SUB_DIM = 16;
const EMB = (v: number[]) => v.slice(0, SUB_DIM);
const subsetNames = new Set(subset.map((s) => s.name));

const nodes: GNNNode[] = subset.map((r) => ({
  name: r.name,
  level: r.level,
  embedding: EMB(r.embedding!),
  children: (noteByName.get(r.name)?.wikilinks ?? []).filter((c) => subsetNames.has(c)),
}));

const rawMap = new Map(nodes.map((n) => [n.name, n.embedding]));
const storedGnnMap = new Map(
  subset.map((r) => [r.name, EMB((r.gnnEmbedding ?? r.embedding)!)]),
);

function cosine(a: number[], b: number[]): number {
  let dot = 0, na = 0, nb = 0;
  for (let i = 0; i < a.length; i++) {
    dot += a[i] * b[i];
    na += a[i] * a[i];
    nb += b[i] * b[i];
  }
  return dot / (Math.sqrt(na) * Math.sqrt(nb) + 1e-9);
}

function avgVec(vecs: number[][]): number[] {
  const out = new Array(vecs[0].length).fill(0);
  for (const v of vecs) for (let i = 0; i < out.length; i++) out[i] += v[i];
  for (let i = 0; i < out.length; i++) out[i] /= vecs.length;
  return out;
}

function avgParentChildCos(emb: Map<string, number[]>): number {
  const vals: number[] = [];
  for (const n of nodes) {
    const p = emb.get(n.name);
    if (!p) continue;
    for (const c of n.children) {
      const ce = emb.get(c);
      if (!ce) continue;
      vals.push(cosine(p, ce));
    }
  }
  if (vals.length === 0) return 0;
  return vals.reduce((s, x) => s + x, 0) / vals.length;
}

const targets = new Map<string, number[]>();
for (const n of nodes) {
  const childEmbs = n.children.map((c) => rawMap.get(c)).filter((v): v is number[] => !!v);
  targets.set(n.name, childEmbs.length > 0 ? avgVec(childEmbs) : n.embedding);
}

const maxLevel = Math.max(...nodes.map((n) => n.level), 0);
const config: GNNConfig = {
  numHeads: 1,
  headDim: 4,
  embDim: SUB_DIM,
  shareLevelWeights: true,
  leakyReluAlpha: 0.2,
};

const blasStatus = getBlasStatus();
console.log(`Backend: ${isBlasAvailable() ? "BLAS" : "JS"} (path=${blasStatus.path ?? "n/a"})`);
console.log(`Subset nodes: ${nodes.length}`);
console.log(`Edges in subset: ${nodes.reduce((s, n) => s + n.children.length, 0)}`);
console.log(`Dims: ${SUB_DIM}, maxLevel=${maxLevel}`);

Subset nodes: 10


Edges in subset: 10


Dims: 16, maxLevel=5


In [2]:
const paramsMessagePassing = initParams(config, maxLevel + 1);
const outMessagePassing = gnnForward(nodes, paramsMessagePassing, config);

const beforeRaw = avgParentChildCos(rawMap);
const afterStoredGnn = avgParentChildCos(storedGnnMap);
const afterRandomMP = avgParentChildCos(outMessagePassing);

console.log("Avg parent-child cosine:");
console.log(`  Raw embeddings:       ${beforeRaw.toFixed(4)}`);
console.log(`  Stored GNN (current): ${afterStoredGnn.toFixed(4)}`);
console.log(`  Random MP only:       ${afterRandomMP.toFixed(4)}`);

Avg parent-child cosine:


  Raw embeddings:       0.5682


  Stored GNN (current): 0.2933


  Random MP only:       0.3406


In [3]:
const paramsTrained = initParams(config, maxLevel + 1);
const LR = 0.03;
const STEPS = 25;
const losses: Array<{step: number, loss: number}> = [];

for (let step = 1; step <= STEPS; step++) {
  const { loss } = gnnTrainStep(nodes, paramsTrained, config, targets, LR);
  losses.push({ step, loss });
}

const outTrained = gnnForward(nodes, paramsTrained, config);
const afterBackprop = avgParentChildCos(outTrained);

console.log(`Final loss: ${losses[losses.length - 1].loss.toFixed(6)}`);
console.log(`Parent-child cosine after backprop: ${afterBackprop.toFixed(4)}`);console.log(`Delta vs raw: ${(afterBackprop - beforeRaw).toFixed(4)}`);
console.log(`Delta vs random MP: ${(afterBackprop - afterRandomMP).toFixed(4)}`);


Final loss: 0.010882


Parent-child cosine after backprop: 0.4880


In [4]:
const lossSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "GNN Numeric Backprop Loss",
  "width": 500,
  "height": 240,
  "data": { "values": losses },
  "mark": { "type": "line", "point": true },
  "encoding": {
    "x": { "field": "step", "type": "quantitative" },
    "y": { "field": "loss", "type": "quantitative" },
    "tooltip": [{ "field": "step" }, { "field": "loss", "format": ".6f" }]
  }
};
await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": lossSpec }, { raw: true });

In [5]:
const summary = [
  { variant: "raw", value: beforeRaw },
  { variant: "stored_gnn", value: afterStoredGnn },
  { variant: "random_mp", value: afterRandomMP },
  { variant: "trained_mp_backprop", value: afterBackprop },
];
console.table(summary.map((x) => ({ ...x, value: x.value.toFixed(4) })));

┌───────┬───────────────────────┬──────────┐
│ (idx) │ variant               │ value    │
├───────┼───────────────────────┼──────────┤
│     0 │ "raw"                 │ "0.5682" │
│     1 │ "stored_gnn"          │ "0.2933" │
│     2 │ "random_mp"           │ "0.3406" │
│     3 │ "trained_mp_backprop" │ "0.4880" │
└───────┴───────────────────────┴──────────┘


In [6]:
db.close();
console.log("Done");

Done
